# Stage 2 -- Aggregation: Daily Full Moments

## Input
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet` -- Panel A, ~100 stocks × ~4,656 dates × 192 factors, keyed on `(permno, date)`
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_daily_engineered.parquet` -- Panel C, market-level macro daily factors, keyed on `date`

## Purpose
Extends the daily aggregation from Notebook 01 by computing **five** cap-weighted cross-sectional statistics per stock factor instead of just the mean. The five moments capture how the distribution of each factor across ~100 stocks changes over time, not just the average level. For example: `cwstd(turnover)` measures how dispersed trading activity is across stocks today; `cwskew(dlyretx)` captures whether the cross-section of returns is positively or negatively skewed; `spread(bid_ask_spread)` shows how different the most vs least liquid stocks are.

Steps 1--4 are identical to Notebook 01 (load & trim, handle warmup NaN, winsorise, compute target). Step 5 is the key difference: full moments aggregation.

---

## Pipeline

### Steps 1--4: Load, Trim, NaN, Winsorise, Target
Identical to Notebook 01. Panel A trimmed to 2006-07-03+, NaN left in place for per-factor per-date exclusion, all stock factors cast to `float64` and winsorised at 1st/99th percentile cross-sectionally per date using vectorised `groupby.transform`. Three ISO columns dropped post-winsorisation. Target computed as next-day cap-weighted market return with date-gap guard.

### Step 5: Cap-Weighted Full Moments Aggregation
For each of the ~189 surviving stock factor columns, five statistics are computed per date. All computation uses numpy arrays and pandas `groupby('date').sum()` (C-engine) to avoid Python loops over dates. Progress is printed every 25 factors with elapsed time and remaining estimate.

For each factor the following are computed in sequence:

**cwmean** (cap-weighted mean):
- Valid mask: non-NaN in both cap and factor value
- Weighted sum / cap sum per date

**cwstd** (cap-weighted standard deviation):
- Deviations from cwmean are computed per stock
- `sqrt(Σ(w × dev²) / Σ(w))` per date
- Tiny negative values from floating-point noise clamped to zero before sqrt

**cwskew** (cap-weighted skewness):
- Standardised deviations (z-scores) computed using cwstd; stocks where cwstd < 1e-10 are excluded
- `Σ(w × z³) / Σ(w)` per date

**cwkurt** (cap-weighted kurtosis):
- `Σ(w × z⁴) / Σ(w)` per date (raw kurtosis, not excess; normal distribution = 3.0)

**spread** (p90 - p10, unweighted):
- 90th minus 10th percentile of the raw factor values across stocks per date, computed without cap weighting
- Captures the range between the most extreme stocks

Results for all five moments × all factors are assembled into a single DataFrame in one shot. NaN counts in the aggregated output are reported.

### Step 6: Merge with Macro Daily + Target
Column name conflicts between aggregated stock moments and Panel C macro factors are checked and resolved with a `stock_` prefix if needed. Three-way merge on `date` (inner join with Panel C, left join for target).

**Post-merge drops:** Any `_cwskew` or `_cwkurt` columns that contain any NaN are dropped. These arise for factors where `cwstd ≈ 0` (the cross-section is near-constant on some dates), making skewness and kurtosis undefined. This is a targeted clean-up rather than a pre-specified drop list.

**Warmup trim:** First 50 rows dropped for Panel C rolling feature warmup. Last row dropped (no target available).

### Step 7: Validation
- No duplicate dates
- NaN count across all feature columns
- Target NaN count
- **Target integrity check:** correlation between `target_daily_return` on date t and `dlyretx_cwmean` on date t+1 (should be ~0.99+)
- **Moment sanity checks:**
  - Minimum `cwstd` across all factors and dates (should be ≥ 0)
  - Minimum and mean `cwkurt` (raw kurtosis; mean ~3 for normal-like distributions, >3 indicates fat tails)
  - Minimum `spread` (should be ≥ 0)
- Column breakdown: stock moment columns, macro factors, target, date

### Step 8: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **Five moments per factor** rather than just the mean, to capture distributional dynamics across the ~100-stock cross-section.
- **cwkurt is raw kurtosis** (not excess), so a normal distribution scores ~3.0. Values above 3 indicate fat tails in the cross-section.
- **spread is unweighted** (p90 - p10 of raw values), in contrast to the four cap-weighted moments. This captures extreme-stock behaviour regardless of market cap.
- **Skew/kurt columns with any NaN are dropped post-merge** rather than pre-specified, since which factors produce undefined higher moments depends on the data.
- **Vectorised aggregation:** for each factor, five separate `groupby().sum()` calls are made, each operating on pre-computed numpy arrays. This avoids Python loops over dates while computing all five moments correctly.
- All other design decisions (start date, winsorisation, ISO drops, 50-row warmup trim, target computation) are identical to Notebook 01.

## Output
`Data/Data_Collection/Final/Stage_2/agg_market_daily_full_moments.parquet` -- keyed on `date`, containing five cap-weighted cross-sectional moments (cwmean, cwstd, cwskew, cwkurt, spread) for each stock daily factor, plus all Panel C macro factors and `target_daily_return`

In [5]:
# %% [markdown]
# # Stage 2 — Aggregation: Monthly Means
#
# Transforms Panel B (stock monthly, ~100 stocks × ~222 months × 197 factors)
# into a single market-level monthly time series using cap-weighted means.
# Merges with Panel D (macro monthly) and a monthly target variable.
#
# Pipeline:
#   1. Load Panel B, Panel D, and Panel A (daily, for target computation)
#   2. Trim all to ≥ 2006-07-31
#   3. Compute monthly target: next-month cap-weighted market return
#   4. Winsorise Panel B stock factors at 1st/99th per month
#   5. Aggregate: cap-weighted mean per month
#   6. Merge aggregated stock means + macro monthly + target
#   7. Trim warmup rows + drop last row (no next-month return)
#   8. Validate
#   9. Save
#
# Input:
#   Panel B: Stage_1_5/.../panel_stock_monthly_engineered.parquet
#   Panel D: Stage_1_5/.../panel_macro_monthly_engineered.parquet
#   Panel A: Stage_1_5/.../panel_stock_daily_engineered.parquet (for target)
#
# Output:
#   Stage_2/agg_market_monthly_means.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import time

PANEL_B_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_monthly_engineered.parquet')
PANEL_D_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_monthly_engineered.parquet')
PANEL_A_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD & TRIM
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD & TRIM")
print("=" * 90)

START_DATE_MONTHLY = '2004-01-31'
START_DATE_DAILY = '2004-01-02'
MIN_STOCKS = 50

# Load Panel B (stock monthly)
panel_b = pd.read_parquet(PANEL_B_PATH)
panel_b['date'] = pd.to_datetime(panel_b['date'])
print(f"\n  Panel B loaded: {panel_b.shape[0]:,} rows × {panel_b.shape[1]} columns")
print(f"    Date range: {panel_b['date'].min().date()} → {panel_b['date'].max().date()}")

panel_b = panel_b[panel_b['date'] >= START_DATE_MONTHLY].reset_index(drop=True)
print(f"    After trim:  {panel_b.shape[0]:,} rows")
print(f"    Date range: {panel_b['date'].min().date()} → {panel_b['date'].max().date()}")
print(f"    Unique months: {panel_b['date'].nunique()}")
print(f"    Avg stocks/month: {panel_b.groupby('date').size().mean():.1f}")

# Load Panel D (macro monthly)
panel_d = pd.read_parquet(PANEL_D_PATH)
panel_d['date'] = pd.to_datetime(panel_d['date'])
print(f"\n  Panel D loaded: {panel_d.shape[0]:,} rows × {panel_d.shape[1]} columns")
print(f"    Date range: {panel_d['date'].min().date()} → {panel_d['date'].max().date()}")

# Load Panel A (daily, for monthly target computation)
print(f"\n  Loading Panel A (daily) for monthly target computation...")
panel_a = pd.read_parquet(PANEL_A_PATH, columns=['permno', 'date', 'dlyret', 'dlycap'])
panel_a['date'] = pd.to_datetime(panel_a['date'])
panel_a = panel_a[panel_a['date'] >= START_DATE_DAILY].reset_index(drop=True)
print(f"    Panel A loaded: {panel_a.shape[0]:,} rows (daily, trimmed)")

# Identify columns
meta_cols = ['permno', 'date', 'month_end_cap']
factor_cols = [c for c in panel_b.columns if c not in meta_cols]
macro_factor_cols = [c for c in panel_d.columns if c != 'date']

print(f"\n  Panel B factor columns: {len(factor_cols)}")
print(f"  Panel D factor columns: {len(macro_factor_cols)}")

# Date alignment
b_dates = set(panel_b['date'].unique())
d_dates = set(panel_d['date'].unique())
common_dates = b_dates & d_dates
only_b = b_dates - d_dates
only_d = d_dates - b_dates

print(f"\n  Date alignment:")
print(f"    Common dates: {len(common_dates)}")
print(f"    Only in Panel B: {len(only_b)}")
print(f"    Only in Panel D: {len(only_d)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: COMPUTE MONTHLY TARGET FROM DAILY RETURNS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: COMPUTE MONTHLY TARGET (next-month cap-weighted market return)")
print("=" * 90)

# Strategy:
# 1. For each stock-month, compound daily returns: monthly_ret = Π(1+r_daily) - 1
# 2. Get month-end cap from Panel B (last day of month per stock)
# 3. For month t: target = Σ(cap_{i,t} × monthly_ret_{i,t+1}) / Σ(cap_{i,t})
# This uses this month's cap (known) × next month's return (unknown).

# Step 2a: compute monthly returns per stock by compounding daily returns
panel_a['_ym'] = panel_a['date'].dt.to_period('M')
panel_a = panel_a.sort_values(['permno', 'date'])

monthly_ret = panel_a.groupby(['permno', '_ym'])['dlyret'].apply(
    lambda x: (1 + x).prod() - 1
).reset_index()
monthly_ret.columns = ['permno', '_ym', 'monthly_ret']

# Convert _ym to month-end date to match Panel B dates
monthly_ret['date'] = monthly_ret['_ym'].dt.to_timestamp('M')

print(f"\n  Monthly returns computed: {len(monthly_ret):,} stock-months")
print(f"    Mean: {monthly_ret['monthly_ret'].mean():.6f}")
print(f"    Std:  {monthly_ret['monthly_ret'].std():.6f}")

# Step 2b: get month-end cap from Panel B
cap_lookup = panel_b[['permno', 'date', 'month_end_cap']].copy()

# Step 2c: for each stock, get NEXT month's return
monthly_ret = monthly_ret.sort_values(['permno', 'date'])
monthly_ret['next_month_ret'] = monthly_ret.groupby('permno')['monthly_ret'].shift(-1)

# Date-gap guard: null out where months are not consecutive
monthly_ret['date_diff'] = monthly_ret.groupby('permno')['date'].diff(-1).abs()
monthly_ret.loc[monthly_ret['date_diff'] > pd.Timedelta(days=45), 'next_month_ret'] = np.nan

# Step 2d: merge cap with next-month return
target_data = cap_lookup.merge(
    monthly_ret[['permno', 'date', 'next_month_ret']],
    on=['permno', 'date'],
    how='inner'
)

# Step 2e: cap-weighted aggregation of next-month return
target_data = target_data.dropna(subset=['next_month_ret', 'month_end_cap'])
target_data['weighted_ret'] = target_data['month_end_cap'] * target_data['next_month_ret']

target_agg = target_data.groupby('date').agg(
    target_monthly_return=('weighted_ret', 'sum'),
    total_cap=('month_end_cap', 'sum'),
).reset_index()
target_agg['target_monthly_return'] = target_agg['target_monthly_return'] / target_agg['total_cap']
target_agg = target_agg[['date', 'target_monthly_return']]

print(f"\n  Monthly target computed: {len(target_agg)} months")
print(f"    Mean:  {target_agg['target_monthly_return'].mean():.6f}")
print(f"    Std:   {target_agg['target_monthly_return'].std():.6f}")
print(f"    Min:   {target_agg['target_monthly_return'].min():.6f}")
print(f"    Max:   {target_agg['target_monthly_return'].max():.6f}")
print(f"    NaN:   {target_agg['target_monthly_return'].isna().sum()}")

# Free memory — Panel A daily no longer needed
del panel_a
import gc; gc.collect()

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: WINSORISE PANEL B STOCK FACTORS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: WINSORISE STOCK FACTORS (1st/99th per month)")
print("=" * 90)

t0 = time.time()

for c in ['permno', 'date', 'month_end_cap']:
    assert c not in factor_cols, f"FATAL: {c} is in factor_cols!"

# Cast to float64
panel_b[factor_cols] = panel_b[factor_cols].astype('float64')

# Vectorised winsorisation
for col in factor_cols:
    p01 = panel_b.groupby('date')[col].transform('quantile', 0.01)
    p99 = panel_b.groupby('date')[col].transform('quantile', 0.99)
    panel_b[col] = panel_b[col].clip(lower=p01, upper=p99)

elapsed = time.time() - t0
print(f"\n  Winsorised {len(factor_cols)} factors in {elapsed:.1f}s")
print(f"  ✓ Winsorisation complete")


# FY1 rollover: FY1 rolls between January and February for a December fiscal
# year-end, so a month-on-month revision comparison spans two different forecast
# targets -- there is no revision to measure, because the forecast target was
# replaced. IBES nulls it. Zero is the honest value: these are DIFFERENCES, so
# "no information" means the change is zero, not that January's change recurred
# (which is what a forward-fill would assert).
ROLLOVER_FACTORS = ['rev_revision_1m', 'rev_revision_3m', 'rev_numest_chg',
                    'rev_fy2_revision_1m', 'rev_eps_divergence',
                    'rev_revision_accel', 'ptg_rev_alignment',
                    'AnnouncementReturn']
for c in ROLLOVER_FACTORS:
    if c in panel_b.columns:
        panel_b[c] = panel_b[c].fillna(0.0)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: CAP-WEIGHTED MEAN AGGREGATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: CAP-WEIGHTED MEAN AGGREGATION")
print("=" * 90)

t0 = time.time()

agg_factors = factor_cols.copy()
cap_arr = panel_b['month_end_cap'].to_numpy(dtype='float64', na_value=np.nan)
date_arr = panel_b['date'].values
sorted_dates = np.sort(panel_b['date'].unique())

print(f"\n  Aggregating {len(agg_factors)} factors across {len(sorted_dates)} months...")

agg_results = {}
for i, col in enumerate(agg_factors):
    vals = panel_b[col].to_numpy(dtype='float64', na_value=np.nan)
    
    valid = ~(np.isnan(vals) | np.isnan(cap_arr))
    weighted = np.where(valid, cap_arr * vals, 0.0)
    cap_valid = np.where(valid, cap_arr, 0.0)
    
    temp = pd.DataFrame({'date': date_arr, 'wv': weighted, 'wc': cap_valid,
                         'n': valid.astype('int64')})
    agg = temp.groupby('date', sort=True).sum()
    cwmean = (agg['wv'] / agg['wc'].replace(0, np.nan)).values
    agg_results[col] = np.where(agg['n'].values >= MIN_STOCKS, cwmean, np.nan)

# Build DataFrame in one shot
agg_results['date'] = sorted_dates
agg_stock = pd.DataFrame(agg_results)

elapsed = time.time() - t0
print(f"\n  Aggregated in {elapsed:.1f}s")
print(f"  Result: {agg_stock.shape[0]} rows × {agg_stock.shape[1]} columns")

# NaN check
agg_nan = agg_stock[agg_factors].isna().sum()
agg_nan_cols = agg_nan[agg_nan > 0]
if len(agg_nan_cols) > 0:
    print(f"\n  Factors with NaN after aggregation: {len(agg_nan_cols)}")
    for c in agg_nan_cols.sort_values(ascending=False).head(25).index:
        n = int(agg_nan_cols[c])
        print(f"    {c:<45s} {n:>5d} NaN ({n / len(agg_stock) * 100:5.1f}%)")

    # Flag anything MIN_STOCKS may have gutted rather than merely trimmed
    gutted = agg_nan_cols[agg_nan_cols > 0.5 * len(agg_stock)]
    if len(gutted):
        print(f"\n  ** {len(gutted)} factors NaN on >50% of dates -- check "
              f"whether MIN_STOCKS={MIN_STOCKS} is too high for these **")
        for c in gutted.index:
            print(f"    {c}")
else:
    print(f"  ✓ Zero NaN in aggregated stock factors")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: MERGE WITH MACRO MONTHLY + TARGET
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: MERGE AGGREGATED STOCK + MACRO MONTHLY + TARGET")
print("=" * 90)

# Check for column name conflicts
stock_cols_set = set(agg_stock.columns) - {'date'}
macro_cols_set = set(panel_d.columns) - {'date'}
overlap = stock_cols_set & macro_cols_set

if overlap:
    print(f"\n  ⚠ Column name conflicts ({len(overlap)}):")
    for c in sorted(overlap):
        print(f"    {c}")
    print(f"    Adding 'stock_' prefix to conflicting stock columns...")
    rename_map = {c: f'stock_{c}' for c in overlap}
    agg_stock = agg_stock.rename(columns=rename_map)
    agg_factors = [rename_map.get(c, c) for c in agg_factors]
else:
    print(f"\n  ✓ No column name conflicts between stock and macro")

# Merge
result = agg_stock.merge(panel_d, on='date', how='inner')
result = result.merge(target_agg, on='date', how='left')

print(f"\n  After merge: {result.shape[0]} rows × {result.shape[1]} columns")
print(f"  Date range: {result['date'].min().date()} → {result['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: TRIM WARMUP + DROP LAST ROW
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: TRIM WARMUP + DROP LAST ROW")
print("=" * 90)

# Trim first 3 rows for Panel D's 3-month rolling warmup
pre_trim = len(result)
result = result.iloc[3:].reset_index(drop=True)
print(f"\n  Trimmed first 3 rows for warmup: {pre_trim} → {len(result)}")

# Drop last row (no next-month return)
result = result.dropna(subset=['target_monthly_return']).reset_index(drop=True)
print(f"  Dropped rows without target: {len(result)} rows remaining")


# Drop OAP factors that stopped publishing in late 2023/2024
oap_drop = ['OptionVolume1', 'OptionVolume2', 
            'PriceDelayRsq', 'PriceDelaySlope', 'PriceDelayTstat']
oap_drop = [c for c in oap_drop if c in result.columns]
result = result.drop(columns=oap_drop)
print(f"Dropped {len(oap_drop)} discontinued OAP factors: {oap_drop}")


print(f"  Date range: {result['date'].min().date()} → {result['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 7: VALIDATE")
print("=" * 90)

# 7a. Duplicate dates
n_dupes = result['date'].duplicated().sum()
assert n_dupes == 0, "FATAL: Duplicate dates!"
print(f"\n  ✓ No duplicate dates")

# 7b. NaN in features
feature_cols = [c for c in result.columns if c not in ['date', 'target_monthly_return']]
feature_nan = result[feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"\n  Feature NaN: {feature_nan_total}")
    print(f"  Columns with NaN ({len(nan_cols)}):")
    for c in nan_cols.head(15).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"  ✓ Zero NaN in features")



# ── 7b2. Trailing NaN ───────────────────────────────────────────────────────
# A factor that stops publishing leaves the END of the sample empty, which lands
# in the test period. Split_D tests 2023-2024. The 5 discontinued OAP factors are
# dropped by name in notebooks 03/04, so anything listed here is new.
TRAILING_TOLERANCE = 6        # ~6 months of trading days

n_rows = len(result)
stopped = {}
for c in feature_cols:
    lv = result[c].last_valid_index()
    if lv is None:
        stopped[c] = ('never valid', n_rows)
    elif n_rows - 1 - lv > TRAILING_TOLERANCE:
        stopped[c] = (str(result['date'].iloc[lv].date()), n_rows - 1 - lv)

if stopped:
    print(f"\n  ** {len(stopped)} columns stop >{TRAILING_TOLERANCE} rows "
          f"before {result['date'].max().date()} **")
    print(f"  {'Column':<45s} {'Last valid':>12s} {'Rows missing':>13s}")
    for c, (d, g) in sorted(stopped.items(), key=lambda x: -x[1][1]):
        print(f"  {c:<45s} {d:>12s} {g:>13d}")
else:
    print(f"  ✓ No columns stop early")

# 7c. Target NaN
target_nan = result['target_monthly_return'].isna().sum()
print(f"  Target NaN: {target_nan} (expect 0)")

# 7d. Target sanity
print(f"\n  Target statistics:")
print(f"    Mean:   {result['target_monthly_return'].mean():.6f} (expect small positive ~0.007)")
print(f"    Std:    {result['target_monthly_return'].std():.6f} (expect ~0.04-0.05)")
print(f"    Min:    {result['target_monthly_return'].min():.6f}")
print(f"    Max:    {result['target_monthly_return'].max():.6f}")
print(f"    Sharpe: {result['target_monthly_return'].mean() / result['target_monthly_return'].std() * np.sqrt(12):.2f} (annualised)")

# 7e. Target integrity check
# The monthly target should be close to compounding the daily targets within each month
# We can't easily verify here without the daily target, but check basic properties
pos_pct = (result['target_monthly_return'] > 0).mean() * 100
print(f"    Positive months: {pos_pct:.1f}% (expect ~55-60%)")

# 7f. Column breakdown
stock_feature_count = len([c for c in result.columns if c in agg_factors])
macro_feature_count = len([c for c in result.columns if c in macro_factor_cols])
print(f"\n  Column breakdown:")
print(f"    Stock cwmean factors: {stock_feature_count}")
print(f"    Macro factors:        {macro_feature_count}")
print(f"    Target:               1")
print(f"    Date:                 1")
print(f"    Total:                {result.shape[1]}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 8: SAVE")
print("=" * 90)

result = result.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'agg_market_monthly_means.parquet'
result.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {result.shape[0]} rows × {result.shape[1]} columns")
print(f"    Size: {file_size / 1e3:.1f} KB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("MONTHLY MEANS AGGREGATION COMPLETE")
print("=" * 90)

print(f"""
  Pipeline:
    Panel B ({panel_b.shape[0]:,} stock-months) → winsorise → cwmean → {stock_feature_count} factors
    Panel D ({len(panel_d)} months) → {macro_feature_count} factors
    Target: next-month cap-weighted market return (compounded from daily)
    Merge: inner join on date

  Result:
    Rows:    {result.shape[0]} months
    Columns: {result.shape[1]} ({stock_feature_count} stock + {macro_feature_count} macro + target + date)
    Dates:   {result['date'].min().date()} → {result['date'].max().date()}
    NaN:     {feature_nan_total} features + {target_nan} target

  Saved: {out_path}

  Next: 04_build_agg_monthly_full_moments.ipynb (adds cwstd, cwskew, cwkurt, spread)
""")

STEP 1: LOAD & TRIM

  Panel B loaded: 25,194 rows × 200 columns
    Date range: 2004-01-31 → 2024-12-31
    After trim:  25,194 rows
    Date range: 2004-01-31 → 2024-12-31
    Unique months: 252
    Avg stocks/month: 100.0

  Panel D loaded: 252 rows × 134 columns
    Date range: 2004-01-31 → 2024-12-31

  Loading Panel A (daily) for monthly target computation...
    Panel A loaded: 525,957 rows (daily, trimmed)

  Panel B factor columns: 197
  Panel D factor columns: 133

  Date alignment:
    Common dates: 252
    Only in Panel B: 0
    Only in Panel D: 0

STEP 2: COMPUTE MONTHLY TARGET (next-month cap-weighted market return)

  Monthly returns computed: 25,086 stock-months
    Mean: 0.008939
    Std:  0.077264

  Monthly target computed: 251 months
    Mean:  0.009208
    Std:   0.041903
    Min:   -0.147255
    Max:   0.129155
    NaN:   0

STEP 3: WINSORISE STOCK FACTORS (1st/99th per month)

  Winsorised 197 factors in 1.2s
  ✓ Winsorisation complete

STEP 4: CAP-WEIGHTED MEAN 

In [2]:
feat = [c for c in result.columns if c not in ('date', 'target_monthly_return')]
nan = result[feat].isna()
pct = (nan.groupby(result['date'].dt.year).mean() * 100).T
pct = pct.loc[pct.max(axis=1) > 0]
print(pct.round(0).astype(int).to_string())

date                 2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024
AnnouncementReturn      0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     9
rev_revision_1m         0     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     9
rev_revision_3m        11    25    25    25    25    25    25    25    25    25    25    25    25    25    25    25    25    25    25    25    27
rev_numest_chg          0     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     9
rev_fy2_revision_1m     0     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     9
rev_eps_divergence      0     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8     8   

In [3]:
pct_m = (nan.groupby(result['date'].dt.month).mean() * 100).T
pct_m = pct_m.loc[pct_m.max(axis=1) > 0]
print(pct_m.round(0).astype(int).to_string())

date                 1    2    3    4   5   6   7   8   9   10  11  12
AnnouncementReturn    0    0    0    0   0   0   0   0   0   0   5   0
rev_revision_1m       0  100    0    0   0   0   0   0   0   0   0   0
rev_revision_3m       0  100  100  100   0   0   0   0   0   0   0   0
rev_numest_chg        0  100    0    0   0   0   0   0   0   0   0   0
rev_fy2_revision_1m   0  100    0    0   0   0   0   0   0   0   0   0
rev_eps_divergence    0  100    0    0   0   0   0   0   0   0   0   0
rev_revision_accel    0  100  100  100   0   0   0   0   0   0   0   0
ptg_rev_alignment     0  100    0    0   0   0   0   0   0   0   0   0
avg_hourly_earnings  10   10   10   14  10  10  10  10  10  10  10  10


In [4]:
print(result.loc[result['AnnouncementReturn'].isna(), 'date'])

247   2024-11-30
Name: date, dtype: datetime64[ns]
